In [1]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw
import ipywidgets as widgets
import numpy as np

In [2]:
box = Box((0, 0, 0), (30, 6, 10))
box.faces.name = "outer"
cyl = sum([Cylinder((5 + (10 * i), 0, 5), Y, 2.5, 8) for i in range(3)])
cyl.faces.name = "cyl"

geo = box - cyl
geo.faces.Min(X).name = "fix"
geo.faces.Max(X).name = "force"

cylboxedges = geo.faces["outer"].edges * geo.faces["cyl"].edges
cylboxedges.name = "cylbox"
geo = geo.MakeChamfer(cylboxedges, 0.3)

mesh = Mesh(OCCGeometry(geo).GenerateMesh(maxh=4)).Curve(2)

ea = {"euler_angles": (30, -30, 0)}
Draw(mesh, **ea)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (3…

BaseWebGuiScene

In [4]:
# Material Parameters
E = 21e+3
nu = 0.35
mu = E / 2 / (1 + nu)
lam = E * nu / ((1 + nu) * (1 - 2 * nu))

# Constant Loading
# force = CoefficientFunction((0, 10, 0))

# Gravity Loading
rho = 1100e-6
g = 9.81e+3
fgrav = -(rho * g)
force = CoefficientFunction((0, fgrav, 0))

In [20]:
tau = 0.01
tend = 20

In [23]:
fes = VectorH1(mesh, order=3, dirichlet="fix")
u, v = fes.TnT()

# Components for Strain Energy
I = Id(mesh.dim)
F = I + Grad(u)
C = F.trans * F
E = 0.5 * (C - I)

def Pow(a, b):
    return a**b  # exp (log(a)*b)

def NeoHooke(C):
    return 0.5 * mu * (Trace(C - I) + 2 * mu / lam * Pow(Det(C), -lam / 2 / mu) - 1)

factor = Parameter(0)

gfu = GridFunction(fes)
gfv = GridFunction(fes)
gfa = GridFunction(fes)
gfuold = GridFunction(fes)
gfvold = GridFunction(fes)
gfaold = GridFunction(fes)

bfa = BilinearForm(fes)
bfa += Variation(NeoHooke(C) * dx)

vel_new = 2 / tau * (u - gfuold) - gfvold
acc_new = 2 / tau * (vel_new - gfvold) - gfaold

bfa += acc_new * v * dx
bfa += -force * v * dx
bfa += rho * acc_new * v * dx
bfa += 0.01 * rho * vel_new * v * dx

In [24]:
gfu_history = GridFunction(fes, multidim=0)
sceneu = Draw(
    gfu,
    deformation=True,
    settings={
        "camera": {"transformations": [{"type": "move", "dir": (0, 0, 1), "dist": -2}]}
    },
)
# scenev = Draw(gfv)
gfu.vec[:] = 0
t = 0
step = 0

tw = widgets.Text(value="t = 0")
display(tw)

while t < tend:
    t += tau
    step += 1
    solvers.Newton(a=bfa, u=gfu, printing=False, inverse="sparsecholesky")

    gfv.vec[:] = 2 / tau * (gfu.vec - gfuold.vec) - gfvold.vec
    gfa.vec[:] = 2 / tau * (gfv.vec - gfvold.vec) - gfaold.vec

    sceneu.Redraw()
    # scenev.Redraw()

    gfuold.vec[:] = gfu.vec
    gfvold.vec[:] = gfv.vec
    gfaold.vec[:] = gfa.vec
    if step % 30 == 0:
        gfu_history.AddMultiDimComponent(gfu.vec)
    tw.value = f"t = {t}"

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'camera': {'transformations':…

Text(value='t = 0')

KeyboardInterrupt: 

In [ ]:
Draw(
    gfu_history,
    mesh,
    interpolate_multidim=True,
    animate=True,
    min=0,
    max=1,
    autoscale=False,
    deformation=True,
    settings={
        "camera": {"transformations": [{"type": "move", "dir": (0, 0, 1), "dist": -2}]}
    },
);